# Hyperparameter Tuning for t-SNE, UMAP, VAE and DBSCAN

This notebook tunes the dimensionality-reduction and DBSCAN parameters used for the replication of Ju and Dubin.

Three pipelines are evaluated:

1. t-SNE + DBSCAN
2. UMAP + DBSCAN
3. VAE + DBSCAN

Each parameter combination is compared with the clustering results reported by Ju and Dubin using:

- Number of clusters
- Silhouette Score
- Davies-Bouldin Index
- Calinski-Harabasz Index

The purpose is to identify reasonable parameter combinations that produce results close to the published study while maintaining a reproducible analysis.

## Setup

### Important Required Libraries

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd

# t-SNE
from sklearn.manifold import TSNE

# UMAP
import umap.umap_ as umap

# DBSCAN
from sklearn.cluster import DBSCAN

# Internal clustering evaluation metrics
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# PyTorch for the Variational Autoencoder
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

#### Observation :
- This block imports all the libraries needed for data handling, dimensionality reduction, clustering, model training, and evaluation.
- `numpy` and `pandas` are used to work with the dataset, while `Path` helps manage file locations.
- `t-SNE` and `UMAP` are imported to reduce high-dimensional data into a smaller representation before clustering.
- `DBSCAN` is used to identify natural groups in the reduced data without fixing the number of clusters beforehand.
- Silhouette Score, Davies–Bouldin Index, and Calinski–Harabasz Index are used to compare how well different parameter settings form clusters.
- PyTorch is included to build and train the VAE model, which provides another way to create a lower-dimensional representation of the data.
- Overall, this block prepares everything needed to tune and compare the t-SNE, UMAP and VAE-based DBSCAN clustering approaches.

### Set Random Seed

In [2]:
# Fixed seed makes the parameter search more reproducible
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

torch.manual_seed(RANDOM_STATE)

# Use the same seed for GPU operations if CUDA is available
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print("Random state:", RANDOM_STATE)

Random state: 42


#### Observation :
- This block sets the random seed to $42$ for Python, NumPy, and PyTorch.
- Using the same seed helps ensure that random operations produce consistent results across repeated runs.
- If a GPU is available, the same seed is also applied to CUDA operations.
- This is especially important for hyperparameter tuning and VAE training, where randomness can otherwise lead to slightly different results each time.
- Overall, this block improves the reproducibility of the analysis, making the comparison between different parameter combinations more reliable.

### Select PyTorch Device

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch device:", device)

PyTorch device: cuda


#### Observation :
- This block checks whether a GPU is available for PyTorch.
- If CUDA is available, the computation is assigned to the GPU; otherwise, it uses the CPU.
- The selected device is stored in the device variable so that the `VAE` model and its data can later be placed on the same hardware.
- Using a GPU can significantly speed up VAE training and hyperparameter testing, especially with larger datasets.
- Overall, this block automatically selects the available computing hardware for efficient VAE model training.

### Set Project Folder Paths

In [4]:
ROOT = Path.cwd().parents[1]

# Prepared dataset produced in the previous notebook
PREP_RESULTS_DIR = (
    ROOT / "results" / "module_2" / "dataset_creation"
)

# All hyperparameter tuning outputs will be stored here
RESULTS_DIR = (
    ROOT
    / "results"
    / "module_2"
    / "hyperparameter_tuning"
)

# Create the results folder if it does not already exist
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Project root:", ROOT.resolve())
print(
    "Dataset preparation folder:",
    PREP_RESULTS_DIR.resolve()
)
print(
    "Hyperparameter tuning results folder:",
    RESULTS_DIR.resolve()
)

Project root: C:\Users\samsa\Documents\ICU Clustering
Dataset preparation folder: C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation
Hyperparameter tuning results folder: C:\Users\samsa\Documents\ICU Clustering\results\module_2\hyperparameter_tuning


#### Observation :
- This block defines the main project folders used throughout the notebook.
- `ROOT` identifies the project’s root directory so file paths can be built consistently.
- `PREP_RESULTS_DIR` points to the folder containing the prepared dataset from the dataset-creation notebook.
- `RESULTS_DIR` defines where all hyperparameter-tuning results will be saved.
- `mkdir(parents = True, exist_ok = True)` creates the output folder if it does not already exist, preventing errors when results are saved later.
- The final print statements display the resolved paths, making it easier to verify that the notebook is reading from and writing to the correct locations.
- Overall, this block organizes the input and output locations so the hyperparameter-tuning process uses the correct prepared data and stores its results consistently.

## Load the Prepared Clustering Dataset

### Load the Dataset

In [5]:
# Change only this filename if your previous notebook
# saved the clustering dataset under another name
CLUSTERING_DATA_PATH = (
    PREP_RESULTS_DIR / "feature_data_scaled.csv"
)

# Load the prepared patient-level clustering matrix
clustering_df = pd.read_csv(
    CLUSTERING_DATA_PATH
)

print("Dataset shape:", clustering_df.shape)

clustering_df.head()

Dataset shape: (7648, 81)


,patientunitstayid,age,gender,admissionheight,admissionweight,min_BUN,min_Hgb,min_WBC x 1000,min_anion gap,min_bicarbonate,...,ethnicity_Hispanic,ethnicity_Native American,ethnicity_Other/Unknown,unittype_CCU-CTICU,unittype_CTICU,unittype_Cardiac ICU,unittype_MICU,unittype_Med-Surg ICU,unittype_Neuro ICU,unittype_SICU
0,151900,0.666667,1.0,0.271546,0.077035,0.055777,0.270588,0.028740,0.339286,0.512195,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,210208,0.472222,0.0,0.267434,0.058982,0.039841,0.476471,0.016535,0.321429,0.512195,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,179269,0.888889,1.0,0.254770,0.061000,0.183267,0.400000,0.023228,0.482143,0.439024,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,172764,0.736111,1.0,0.275658,0.075577,0.039841,0.700000,0.073622,0.535714,0.317073,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,166175,0.583333,0.0,0.304934,0.100695,0.023904,0.535294,0.043701,0.428571,0.414634,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


#### Observation :
- This block loads the scaled patient-level dataset created in the previous notebook.
- The file `feature_data_scaled.csv` is taken from the dataset-preparation results folder.
- `pd.read_csv()` loads the data into `clustering_df` for use in the hyperparameter-tuning process.
- The dataset shape is printed to confirm the number of patient records and columns loaded correctly.
- `clustering_df.head()` provides a quick check of the first few records and confirms that the data structure looks correct.
- Overall, this block brings the prepared and scaled healthcare dataset into the notebook so it can be used consistently for t-SNE, UMAP, VAE, and DBSCAN tuning.

### Prepare the Feature Matrix

In [6]:
# Unique eICU ICU-stay identifier
ID_COL = "patientunitstayid"

# Patient ID must not be included as a clustering feature
feature_cols = [
    column
    for column in clustering_df.columns
    if column != ID_COL
]

# All three methods use exactly the same feature matrix
X = clustering_df[
    feature_cols
].to_numpy(dtype=np.float32)

print("Patients:", X.shape[0])
print("Clustering features:", X.shape[1])

Patients: 7648
Clustering features: 80


#### Observation :
- This block identifies `patientunitstayid` as the unique ICU-stay identifier.
- The patient identifier is deliberately excluded from the feature set because it is only an ID and does not contain meaningful clinical information for clustering.
- All remaining columns are selected as the input features.
- These features are converted into the matrix `X`, which will be used by t-SNE, UMAP, and VAE.
- Using the same feature matrix for all three methods keeps the later comparison consistent and fair.
- Overall, this block separates the patient identifier from the clinical features and creates the final feature matrix used for clustering.

### Verify the Feature Matrix

In [7]:
# Dimensionality-reduction methods require complete numerical data
print(
    "Missing values:",
    np.isnan(X).sum()
)

print(
    "Infinite values:",
    np.isinf(X).sum()
)

# The previous notebook used Min-Max scaling,
# so values should be approximately between 0 and 1
print(
    "Minimum feature value:",
    X.min()
)

print(
    "Maximum feature value:",
    X.max()
)

Missing values: 0
Infinite values: 0
Minimum feature value: 0.0
Maximum feature value: 1.0


#### Observation :
- This block creates a reference table containing the published clustering results from Ju and Dubin for the three methods: `t-SNE + DBSCAN`, `UMAP + DBSCAN`, and `VAE + DBSCAN`.
- It records the expected number of clusters and three clustering-quality metrics: Silhouette Score, Davies–Bouldin Index (DBI), and Calinski–Harabasz Index (CHI).
- Among the reported results, VAE + DBSCAN has the highest Silhouette Score ($0.89$) and CHI ($138,637.94$), indicating strong cluster separation and structure.
- UMAP + DBSCAN has the lowest DBI ($0.09$), which indicates very compact and well-separated clusters.
- These values serve as benchmark targets for evaluating how closely the tuned models in this notebook reproduce the published clustering performance.
- Overall, this block establishes the benchmark results that the later hyperparameter-tuning process will use for comparison.

## Define Ju and Dubin Benchmark Results

### Create the Published Results Table

In [8]:
paper_results = pd.DataFrame({
    "method": [
        "t-SNE + DBSCAN",
        "UMAP + DBSCAN",
        "VAE + DBSCAN"
    ],

    "clusters": [
        22,
        22,
        12
    ],

    "silhouette": [
        0.67,
        0.84,
        0.89
    ],

    "dbi": [
        0.25,
        0.09,
        0.19
    ],

    "chi": [
        8576.69,
        54413.23,
        138637.94
    ]
})

paper_results

,method,clusters,silhouette,dbi,chi
0,t-SNE + DBSCAN,22,0.67,0.25,8576.69
1,UMAP + DBSCAN,22,0.84,0.09,54413.23
2,VAE + DBSCAN,12,0.89,0.19,138637.94


#### Observation :
- This block performs a data-quality check before dimensionality reduction and clustering.
- It checks for missing (`NaN`) values and infinite values, because these can cause t-SNE, UMAP, VAE, or DBSCAN to fail or produce unreliable results.
- It also checks the minimum and maximum feature values.
- Since the dataset was previously normalized using Min-Max scaling, most values are expected to fall approximately between $0$ and $1$.
- This confirms that the feature matrix is numerically suitable before hyperparameter tuning begins.

### Store the Published Target Values

In [9]:
# -------------------------
# t-SNE targets
# -------------------------

TSNE_TARGET_CLUSTERS = 22
TSNE_TARGET_SILHOUETTE = 0.67
TSNE_TARGET_DBI = 0.25
TSNE_TARGET_CHI = 8576.69


# -------------------------
# UMAP targets
# -------------------------

UMAP_TARGET_CLUSTERS = 22
UMAP_TARGET_SILHOUETTE = 0.84
UMAP_TARGET_DBI = 0.09
UMAP_TARGET_CHI = 54413.23


# -------------------------
# VAE targets
# -------------------------

VAE_TARGET_CLUSTERS = 12
VAE_TARGET_SILHOUETTE = 0.89
VAE_TARGET_DBI = 0.19
VAE_TARGET_CHI = 138637.94

#### Observation :
- This block converts the published results from Ju and Dubin into target values for each clustering approach.
- For t-SNE + DBSCAN, the target is $22$ clusters with a Silhouette Score of $0.67$, DBI of $0.25$, and CHI of $8576.69$.
- For UMAP + DBSCAN, the target is $22$ clusters with a Silhouette Score of $0.84$, DBI of $0.09$, and CHI of $54413.23$.
- For VAE + DBSCAN, the target is $12$ clusters with a Silhouette Score of $0.89$, DBI of $0.19$, and CHI of $138637.94$.
- These values will later be used to measure how close each tested hyperparameter combination comes to the published results.
- Overall, this block defines the reference targets that guide the hyperparameter selection for all three clustering pipelines.

### Set Silhouette Sample Size

In [10]:
# This sample is used only during hyperparameter tuning.
# The same sample-size rule is applied to all parameter combinations.
SILHOUETTE_SAMPLE_SIZE = 2500

print(
    "Silhouette tuning sample size:",
    SILHOUETTE_SAMPLE_SIZE
)

Silhouette tuning sample size: 2500


#### Observation :
- This block sets the Silhouette Score sample size to $2,500$ patients during hyperparameter tuning.
- Instead of calculating the Silhouette Score on the entire dataset for every parameter combination, it uses a fixed sample to reduce computation time.
- The same sample-size rule is applied across all tested configurations, which keeps the comparison consistent.
- A sample of 2,500 observations provides a practical balance between computational efficiency and a meaningful estimate of cluster quality.
- Overall, this block makes the hyperparameter search faster while keeping Silhouette Score evaluation consistent across the different clustering configurations.

## t-SNE + DBSCAN Hyperparameter Tuning

### Define t-SNE Parameters to Test

In [11]:
tsne_perplexities = [
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80
]

# These parameters remain fixed throughout the search
TSNE_COMPONENTS = 2
TSNE_METRIC = "euclidean"
TSNE_INIT = "pca"
TSNE_LEARNING_RATE = "auto"
TSNE_ITERATIONS = 1000

print(
    "Perplexities:",
    tsne_perplexities
)

Perplexities: [10, 20, 30, 40, 50, 60, 70, 80]


#### Observation :
- This block defines the t-SNE perplexity values that will be tested during hyperparameter tuning, ranging from $10$ to $80$.
- Perplexity controls how t-SNE balances local versus broader neighborhood relationships between patient records.
- Testing several perplexity values helps identify which setting produces a representation that is most suitable for DBSCAN clustering.
- Other t-SNE settings are kept fixed: $2$ output dimensions, Euclidean distance, PCA initialization, automatic learning rate, and $1,000$ iterations.
- Keeping these settings constant ensures that the effect of perplexity can be compared fairly.
- Overall, this block defines the t-SNE search range while keeping the remaining t-SNE settings consistent.

### Define t-SNE DBSCAN Parameters to Test

In [12]:
# Test a focused epsilon range around the region previously used in our t-SNE replication
tsne_eps_values = [
    3.0,
    3.5,
    4.0,
    4.5,
    5.0,
    5.5,
    6.0,
    6.5,
    7.0
]

# Ju and Dubin used MinPts = 50 as their main setting 40 and 60 provide a small sensitivity range.
dbscan_min_samples_values = [
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100
]

print(
    "Epsilon values:",
    tsne_eps_values
)

print(
    "Min samples values:",
    dbscan_min_samples_values
)

Epsilon values: [3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0]
Min samples values: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


#### Observation :
- This block defines the DBSCAN hyperparameter ranges that will be tested with the t-SNE embeddings.
- `eps` values range from $3.0$ to $7.0$, controlling how close patient points must be to be considered part of the same neighborhood.
- `min_samples` values range from $10$ to $100$, controlling how many nearby patients are required for a region to be considered dense enough to form a cluster.
- The search includes $50$, reflecting the MinPts value used by Ju and Dubin, while also testing lower and higher values for sensitivity.
- Testing multiple combinations helps determine which DBSCAN settings produce clusters that most closely match the target clustering results.
- Overall, this block sets the DBSCAN search space for the t-SNE pipeline so the notebook can identify the most suitable `eps` and `min_samples` combination.

### Run t-SNE + DBSCAN Parameter Search

In [13]:
# Store the results from every parameter combination
tsne_tuning_results = []

for perplexity in tsne_perplexities:

    print(
        f"Running t-SNE: perplexity = {perplexity}"
    )

    # Create a new t-SNE representation
    tsne_model = TSNE(
        n_components=TSNE_COMPONENTS,
        perplexity=perplexity,
        metric=TSNE_METRIC,
        init=TSNE_INIT,
        learning_rate=TSNE_LEARNING_RATE,
        max_iter=TSNE_ITERATIONS,
        random_state=RANDOM_STATE
    )

    tsne_embedding = (
        tsne_model.fit_transform(X)
    )

    # Test several DBSCAN configurations
    # using the current t-SNE embedding
    for eps in tsne_eps_values:

        for min_samples in dbscan_min_samples_values:

            dbscan = DBSCAN(
                eps=eps,
                min_samples=min_samples,
                metric="euclidean"
            )

            labels = dbscan.fit_predict(
                tsne_embedding
            )

            # DBSCAN uses -1 for noise
            valid_mask = labels != -1

            valid_embedding = (
                tsne_embedding[valid_mask]
            )

            valid_labels = (
                labels[valid_mask]
            )

            # Count actual clusters
            n_clusters = len(
                np.unique(valid_labels)
            )

            # Count patients classified as DBSCAN noise
            n_outliers = np.sum(
                labels == -1
            )

            outlier_percentage = (
                n_outliers / len(labels)
            ) * 100

            # Internal metrics require at least two clusters
            if n_clusters >= 2:

                try:

                    sample_size = min(
                        SILHOUETTE_SAMPLE_SIZE,
                        len(valid_embedding)
                    )

                    silhouette = silhouette_score(
                        valid_embedding,
                        valid_labels,
                        sample_size=sample_size,
                        random_state=RANDOM_STATE
                    )

                    dbi = davies_bouldin_score(
                        valid_embedding,
                        valid_labels
                    )

                    chi = calinski_harabasz_score(
                        valid_embedding,
                        valid_labels
                    )

                except ValueError:

                    silhouette = np.nan
                    dbi = np.nan
                    chi = np.nan

            else:

                silhouette = np.nan
                dbi = np.nan
                chi = np.nan

            # Save the result
            tsne_tuning_results.append({
                "perplexity": perplexity,
                "epsilon": eps,
                "min_samples": min_samples,
                "clusters": n_clusters,
                "outliers": n_outliers,
                "outlier_percentage": outlier_percentage,
                "silhouette": silhouette,
                "dbi": dbi,
                "chi": chi
            })

Running t-SNE: perplexity = 10
Running t-SNE: perplexity = 20
Running t-SNE: perplexity = 30
Running t-SNE: perplexity = 40
Running t-SNE: perplexity = 50
Running t-SNE: perplexity = 60
Running t-SNE: perplexity = 70
Running t-SNE: perplexity = 80


#### Observation :
- This block runs the complete t-SNE + DBSCAN hyperparameter search.
- For each t-SNE perplexity value, it creates a new $2$-dimensional patient representation.
- Each t-SNE representation is then tested with every combination of DBSCAN `eps` and `min_samples`.
- Patients labeled $-1$ by DBSCAN are treated as noise/outliers and excluded from the clustering-quality calculations.
- The block records the number of clusters and percentage of outliers produced by each parameter combination.
- When at least two valid clusters are present, it calculates the Silhouette Score, Davies–Bouldin Index, and Calinski–Harabasz Index.
- Invalid clustering solutions are assigned NaN values instead of stopping the tuning process.
- Every tested configuration and its results are stored in `tsne_tuning_results` for later comparison and ranking.
- Overall, this block systematically tests t-SNE and DBSCAN parameter combinations and measures how well each combination separates the patient groups.

### Create the t-SNE Results Table

In [14]:
tsne_tuning_df = pd.DataFrame(
    tsne_tuning_results
)

print(
    "Parameter combinations tested:",
    len(tsne_tuning_df)
)

tsne_tuning_df.head()

Parameter combinations tested: 720


,perplexity,epsilon,min_samples,clusters,outliers,outlier_percentage,silhouette,dbi,chi
0,10,3.0,10,60,145,1.895921,0.145549,0.454688,1396.333658
1,10,3.0,20,82,2479,32.413703,0.519351,0.552791,9336.699256
2,10,3.0,30,29,5864,76.673640,0.734219,0.368788,18603.099906
3,10,3.0,40,6,7346,96.051255,0.902799,0.140425,34133.737185
4,10,3.0,50,0,7648,100.000000,NaN,NaN,NaN


#### Observation :
- This block converts all the t-SNE + DBSCAN tuning results collected in the previous step into a pandas DataFrame.
- Each row represents one tested combination of t-SNE perplexity, DBSCAN `eps`, and `min_samples`, together with its clustering results.
- It prints the total number of parameter combinations tested, which helps confirm that the full search was completed.
- `head()` displays the first few results so the structure and stored metrics can be checked.
- Overall, this block organizes the t-SNE tuning results into a structured table so they can be filtered, compared, and ranked in the next steps.

### Calculate t-SNE Similarity to Ju and Dubin

In [15]:
# Remove invalid metric combinations
tsne_tuning_df = tsne_tuning_df.dropna(
    subset=[
        "silhouette",
        "dbi",
        "chi"
    ]
).copy()


# Difference in cluster count
tsne_tuning_df["cluster_error"] = (
    abs(
        tsne_tuning_df["clusters"]
        - TSNE_TARGET_CLUSTERS
    )
    / TSNE_TARGET_CLUSTERS
)


# Difference in Silhouette Score
tsne_tuning_df["silhouette_error"] = (
    abs(
        tsne_tuning_df["silhouette"]
        - TSNE_TARGET_SILHOUETTE
    )
    / TSNE_TARGET_SILHOUETTE
)


# Difference in Davies-Bouldin Index
tsne_tuning_df["dbi_error"] = (
    abs(
        tsne_tuning_df["dbi"]
        - TSNE_TARGET_DBI
    )
    / TSNE_TARGET_DBI
)


# Difference in Calinski-Harabasz Index
tsne_tuning_df["chi_error"] = (
    abs(
        tsne_tuning_df["chi"]
        - TSNE_TARGET_CHI
    )
    / TSNE_TARGET_CHI
)


# Average relative difference across all four targets
tsne_tuning_df["matching_error"] = (
    tsne_tuning_df[
        [
            "cluster_error",
            "silhouette_error",
            "dbi_error",
            "chi_error"
        ]
    ]
    .mean(axis=1)
)

#### Observation :
- This block removes parameter combinations where the clustering metrics could not be calculated.
- It then measures how far each t-SNE + DBSCAN result is from the published Ju and Dubin targets.
- Separate relative errors are calculated for the number of clusters, Silhouette Score, DBI, and CHI.
- Using relative error makes the comparison fair because these metrics have very different numerical scales.
- The individual errors are combined into a single `matching_error`, which represents how closely each parameter combination reproduces the published clustering characteristics.
- A lower matching error indicates a better overall match to the target results.
- Overall, this block creates a common scoring method for comparing all valid t-SNE + DBSCAN parameter combinations against the published benchmark.

### Rank t-SNE Parameter Combinations

In [16]:
# Smaller matching error means the combination is closer to the published Ju and Dubin clustering characteristics
tsne_ranked = (
    tsne_tuning_df
    .sort_values("matching_error")
    .reset_index(drop=True)
)

tsne_ranked[
    [
        "perplexity",
        "epsilon",
        "min_samples",
        "clusters",
        "outlier_percentage",
        "silhouette",
        "dbi",
        "chi",
        "matching_error"
    ]
].head(10).round(4)

,perplexity,epsilon,min_samples,clusters,outlier_percentage,silhouette,dbi,chi,matching_error
0,80,3.0,40,22,4.2626,0.6695,0.2766,8775.6767,0.0326
1,70,3.0,40,22,4.2626,0.6655,0.2763,8269.7121,0.0369
2,70,3.0,50,21,5.1124,0.6652,0.2740,8611.2394,0.0382
3,70,3.5,40,22,3.7395,0.6590,0.2776,8309.6564,0.0395
4,70,3.5,50,21,4.3541,0.6596,0.2760,8604.1097,0.0420
5,80,3.0,50,21,4.9163,0.6734,0.2722,9106.7178,0.0503
6,70,3.0,60,20,6.3938,0.6697,0.2772,9015.9503,0.0628
7,70,3.5,60,20,5.0994,0.6597,0.2796,8930.1273,0.0665
8,80,3.5,40,21,3.5957,0.6710,0.2870,9265.9562,0.0688
9,60,3.0,60,21,8.7866,0.6500,0.2758,7738.7509,0.0690


#### Observation :
- This block ranks all valid t-SNE + DBSCAN parameter combinations using the `matching_error` calculated earlier.
- The combinations are sorted from lowest to highest matching error, so the results closest to the Ju and Dubin benchmark appear first.
- It displays the top $10$ parameter combinations along with their perplexity, `epsilon`, `min_samples`, number of clusters, outlier percentage, and clustering-quality metrics.
- This makes it easier to compare the strongest configurations rather than reviewing every tested combination.
- Overall, this block identifies the t-SNE + DBSCAN settings that most closely reproduce the published clustering characteristics.

### Show the Best t-SNE + DBSCAN Hyperparameters

In [17]:
best_tsne = tsne_ranked.iloc[0]

print(
    "Best t-SNE + DBSCAN Hyperparameters"
)

print("-" * 45)

print(
    "Perplexity:",
    int(best_tsne["perplexity"])
)

print(
    "Epsilon:",
    best_tsne["epsilon"]
)

print(
    "Min samples:",
    int(best_tsne["min_samples"])
)

print()

print("Clustering result")

print(
    "Clusters:",
    int(best_tsne["clusters"])
)

print(
    "Outlier percentage:",
    round(best_tsne["outlier_percentage"], 2),
    "%"
)

print(
    "Silhouette:",
    round(best_tsne["silhouette"], 4)
)

print(
    "Davies-Bouldin Index:",
    round(best_tsne["dbi"], 4)
)

print(
    "Calinski-Harabasz Index:",
    round(best_tsne["chi"], 2)
)

print(
    "Matching error:",
    round(best_tsne["matching_error"], 4)
)

Best t-SNE + DBSCAN Hyperparameters
---------------------------------------------
Perplexity: 80
Epsilon: 3.0
Min samples: 40

Clustering result
Clusters: 22
Outlier percentage: 4.26 %
Silhouette: 0.6695
Davies-Bouldin Index: 0.2766
Calinski-Harabasz Index: 8775.68
Matching error: 0.0326


#### Observation :
- This block selects the best t-SNE + DBSCAN configuration from the ranked tuning results.
- It takes the first row of `tsne_ranked`, which has the lowest matching error compared with the Ju and Dubin benchmark.
- It reports the selected perplexity, DBSCAN epsilon, and minimum samples.
- It also displays the resulting number of clusters, outlier percentage, Silhouette Score, DBI, and CHI.
- This gives a clear summary of the parameter combination that most closely matches the target clustering characteristics.
- Overall, this block identifies and reports the final best-performing t-SNE + DBSCAN hyperparameters from the tuning process.

### Save t-SNE Hyperparameter Tuning Results

In [18]:
# Save every tested t-SNE + DBSCAN parameter combination
tsne_tuning_df.to_csv(
    RESULTS_DIR / "tsne_tuning_results.csv",
    index=False
)

# Save only the 10 parameter combinations
# that are closest to the Ju and Dubin results
tsne_ranked.head(10).to_csv(
    RESULTS_DIR / "tsne_top_parameters.csv",
    index=False
)

print("Saved t-SNE tuning results.")

Saved t-SNE tuning results.


#### Observation :
- This block saves the complete t-SNE + DBSCAN hyperparameter tuning results as a CSV file.
- `tsne_tuning_results.csv` contains all tested parameter combinations and their clustering metrics.
- It also saves the top 10 parameter combinations with the lowest matching error in `tsne_top_parameters.csv`.
- Keeping both files preserves the full experiment while providing a smaller set of the best-performing configurations for later analysis.
- Overall, this block stores the t-SNE tuning results so the selected hyperparameters and their performance can be reused and compared in later stages.

## UMAP + DBSCAN Hyperparameter Tuning

### Define UMAP Parameters to Test

In [19]:
# Controls how many neighboring observations
# UMAP considers when constructing the manifold
umap_neighbors_values = [
    10,
    15,
    20,
    25,
    30,
    35,
    40,
    45,
    50
]

# Controls how closely UMAP allows points
# to be packed in the low-dimensional representation
umap_min_dist_values = [
    0.0,
    0.1,
    0.15,
    0.2,
    0.25,
    0.3,
    0.35,
    0.4,
    0.45,
    0.5
]

UMAP_COMPONENTS = 2
UMAP_METRIC = "euclidean"

print(
    "n_neighbors values:",
    umap_neighbors_values
)

print(
    "min_dist values:",
    umap_min_dist_values
)

n_neighbors values: [10, 15, 20, 25, 30, 35, 40, 45, 50]
min_dist values: [0.0, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]


#### Observation :
- This block defines the UMAP hyperparameters that will be tested during tuning.
- `n_neighbors` ranges from $10$ to $50$ and controls how much local neighborhood information UMAP uses when building the lower-dimensional representation.
- Smaller values focus more on local patient similarities, while larger values capture broader structure in the dataset.
- `min_dist` ranges from $0.0$ to $0.5$ and controls how tightly points can be placed together in the UMAP space.
- Lower `min_dist` values can produce more compact groups, which may help DBSCAN detect clusters.
- UMAP is fixed to 2 dimensions with Euclidean distance, keeping these settings constant while `n_neighbors` and `min_dist` are tuned.
- Overall, this block defines the UMAP search space used to find a low-dimensional representation that produces clear and meaningful patient clusters with DBSCAN.

### Define UMAP DBSCAN Parameters to Test

In [20]:
# UMAP coordinates are on a different numerical scale
# than t-SNE, so epsilon uses its own search range
umap_eps_values = [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70

]

print(
    "UMAP epsilon values:",
    umap_eps_values
)

print(
    "DBSCAN min samples:",
    dbscan_min_samples_values
)

UMAP epsilon values: [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]
DBSCAN min samples: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


#### Observation :
- This block defines the DBSCAN `eps` values specifically for the UMAP representation.
- The tested `eps` values range from $0.30$ to $0.70$.
- UMAP produces coordinates on a different numerical scale than t-SNE, so it requires a smaller and separate epsilon range.
- `eps` determines how close patient points must be for DBSCAN to consider them part of the same neighborhood.
- The previously defined `min_samples` values are reused, allowing different combinations of neighborhood radius and minimum cluster density to be evaluated.
- Overall, this block sets the DBSCAN search range appropriate for the UMAP embedding so suitable patient clusters can be identified.

### Run UMAP + DBSCAN Parameter Search

In [21]:
umap_tuning_results = []

for n_neighbors in umap_neighbors_values:

    for min_dist in umap_min_dist_values:

        print(
            "Running UMAP:",
            f"n_neighbors = {n_neighbors},",
            f"min_dist = {min_dist}"
        )

        # Create the UMAP representation
        umap_model = umap.UMAP(
            n_components=UMAP_COMPONENTS,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            metric=UMAP_METRIC,
            random_state=RANDOM_STATE,
            n_jobs=1
        )

        umap_embedding = (
            umap_model.fit_transform(X)
        )

        # Test DBSCAN settings
        for eps in umap_eps_values:

            for min_samples in dbscan_min_samples_values:

                dbscan = DBSCAN(
                    eps=eps,
                    min_samples=min_samples,
                    metric="euclidean"
                )

                labels = dbscan.fit_predict(
                    umap_embedding
                )

                valid_mask = labels != -1

                valid_embedding = (
                    umap_embedding[valid_mask]
                )

                valid_labels = (
                    labels[valid_mask]
                )

                n_clusters = len(
                    np.unique(valid_labels)
                )

                n_outliers = np.sum(
                    labels == -1
                )

                outlier_percentage = (
                    n_outliers / len(labels)
                ) * 100

                if n_clusters >= 2:

                    try:

                        sample_size = min(
                            SILHOUETTE_SAMPLE_SIZE,
                            len(valid_embedding)
                        )

                        silhouette = silhouette_score(
                            valid_embedding,
                            valid_labels,
                            sample_size=sample_size,
                            random_state=RANDOM_STATE
                        )

                        dbi = davies_bouldin_score(
                            valid_embedding,
                            valid_labels
                        )

                        chi = calinski_harabasz_score(
                            valid_embedding,
                            valid_labels
                        )

                    except ValueError:

                        silhouette = np.nan
                        dbi = np.nan
                        chi = np.nan

                else:

                    silhouette = np.nan
                    dbi = np.nan
                    chi = np.nan

                umap_tuning_results.append({
                    "n_neighbors": n_neighbors,
                    "min_dist": min_dist,
                    "epsilon": eps,
                    "min_samples": min_samples,
                    "clusters": n_clusters,
                    "outliers": n_outliers,
                    "outlier_percentage": outlier_percentage,
                    "silhouette": silhouette,
                    "dbi": dbi,
                    "chi": chi
                })

Running UMAP: n_neighbors = 10, min_dist = 0.0
Running UMAP: n_neighbors = 10, min_dist = 0.1
Running UMAP: n_neighbors = 10, min_dist = 0.15
Running UMAP: n_neighbors = 10, min_dist = 0.2
Running UMAP: n_neighbors = 10, min_dist = 0.25
Running UMAP: n_neighbors = 10, min_dist = 0.3
Running UMAP: n_neighbors = 10, min_dist = 0.35
Running UMAP: n_neighbors = 10, min_dist = 0.4
Running UMAP: n_neighbors = 10, min_dist = 0.45
Running UMAP: n_neighbors = 10, min_dist = 0.5
Running UMAP: n_neighbors = 15, min_dist = 0.0
Running UMAP: n_neighbors = 15, min_dist = 0.1
Running UMAP: n_neighbors = 15, min_dist = 0.15
Running UMAP: n_neighbors = 15, min_dist = 0.2
Running UMAP: n_neighbors = 15, min_dist = 0.25
Running UMAP: n_neighbors = 15, min_dist = 0.3
Running UMAP: n_neighbors = 15, min_dist = 0.35
Running UMAP: n_neighbors = 15, min_dist = 0.4
Running UMAP: n_neighbors = 15, min_dist = 0.45
Running UMAP: n_neighbors = 15, min_dist = 0.5
Running UMAP: n_neighbors = 20, min_dist = 0.0
Runni

#### Observation :
- This block runs the complete UMAP + DBSCAN hyperparameter search.
- For every combination of `n_neighbors` and `min_dist`, UMAP creates a 2-dimensional representation of the patient data.
- Each UMAP representation is then tested with different combinations of DBSCAN `eps` and `min_samples`.
- Patients labeled `-1` by DBSCAN are treated as outliers and excluded from the clustering-quality calculations.
- The block records the number of clusters and percentage of outliers for every parameter combination.
- When at least two clusters are formed, it calculates the Silhouette Score, Davies–Bouldin Index, and Calinski–Harabasz Index.
- If the metrics cannot be calculated, `NaN` values are stored instead of interrupting the tuning process.
- All results are collected in `umap_tuning_results` for later ranking and comparison.
- Overall, this block systematically searches for the UMAP and DBSCAN settings that produce the strongest patient clustering structure.

### Create the UMAP Results Table

In [22]:
umap_tuning_df = pd.DataFrame(
    umap_tuning_results
)

print(
    "Parameter combinations tested:",
    len(umap_tuning_df)
)

umap_tuning_df.head()

Parameter combinations tested: 8100


,n_neighbors,min_dist,epsilon,min_samples,clusters,outliers,outlier_percentage,silhouette,dbi,chi
0,10,0.0,0.3,10,46,33,0.431485,0.886819,0.078169,43342.225560
1,10,0.0,0.3,20,34,207,2.706590,0.895262,0.079516,55425.170988
2,10,0.0,0.3,30,25,407,5.321653,0.902818,0.077706,70790.758811
3,10,0.0,0.3,40,24,438,5.726987,0.905112,0.076311,72998.621984
4,10,0.0,0.3,50,22,522,6.825314,0.909246,0.077243,77257.019112


#### Observation :
- This block converts all UMAP + DBSCAN tuning results into a pandas DataFrame called `umap_tuning_df`.
- Each row represents one tested combination of UMAP and DBSCAN hyperparameters with its corresponding clustering results.
- It prints the total number of parameter combinations tested, confirming the size of the tuning search.
- `head()` displays the first few rows to quickly verify that the results were stored correctly.
- Overall, this block organizes the UMAP tuning results into a structured table so they can be evaluated and ranked in the next steps.

### Calculate UMAP Similarity to Ju and Dubin

In [23]:
umap_tuning_df = umap_tuning_df.dropna(
    subset=[
        "silhouette",
        "dbi",
        "chi"
    ]
).copy()


umap_tuning_df["cluster_error"] = (
    abs(
        umap_tuning_df["clusters"]
        - UMAP_TARGET_CLUSTERS
    )
    / UMAP_TARGET_CLUSTERS
)


umap_tuning_df["silhouette_error"] = (
    abs(
        umap_tuning_df["silhouette"]
        - UMAP_TARGET_SILHOUETTE
    )
    / UMAP_TARGET_SILHOUETTE
)


umap_tuning_df["dbi_error"] = (
    abs(
        umap_tuning_df["dbi"]
        - UMAP_TARGET_DBI
    )
    / UMAP_TARGET_DBI
)


umap_tuning_df["chi_error"] = (
    abs(
        umap_tuning_df["chi"]
        - UMAP_TARGET_CHI
    )
    / UMAP_TARGET_CHI
)


umap_tuning_df["matching_error"] = (
    umap_tuning_df[
        [
            "cluster_error",
            "silhouette_error",
            "dbi_error",
            "chi_error"
        ]
    ]
    .mean(axis=1)
)

#### Observation :
- This block removes UMAP + DBSCAN results where the Silhouette Score, DBI, or CHI could not be calculated.
- It then compares each valid result with the target UMAP clustering results from Ju and Dubin.
- Relative errors are calculated for the number of clusters, Silhouette Score, DBI, and CHI.
- These four errors are averaged into a single `matching_error` value.
- A smaller `matching_error` means the tested parameter combination is closer to the published benchmark.
- Overall, this block creates a single comparison score to identify which UMAP + DBSCAN parameter combination best reproduces the target clustering results.

### Rank UMAP Parameter Combinations

In [24]:
umap_ranked = (
    umap_tuning_df
    .sort_values("matching_error")
    .reset_index(drop=True)
)

umap_ranked[
    [
        "n_neighbors",
        "min_dist",
        "epsilon",
        "min_samples",
        "clusters",
        "outlier_percentage",
        "silhouette",
        "dbi",
        "chi",
        "matching_error"
    ]
].head(10).round(4)

,n_neighbors,min_dist,epsilon,min_samples,clusters,outlier_percentage,silhouette,dbi,chi,matching_error
0,10,0.1,0.35,50,22,6.8384,0.8700,0.1091,48604.1647,0.0886
1,10,0.1,0.45,50,22,6.8253,0.8712,0.1091,48605.4598,0.0890
2,10,0.1,0.40,50,22,6.8253,0.8712,0.1091,48605.4598,0.0890
3,10,0.1,0.50,50,22,6.8253,0.8712,0.1091,48605.4598,0.0890
4,10,0.1,0.55,50,22,6.8122,0.8726,0.1091,48615.3514,0.0895
5,10,0.1,0.70,50,22,6.5900,0.8711,0.1103,48828.1154,0.0913
6,10,0.1,0.65,50,22,6.6684,0.8709,0.1102,48719.2974,0.0915
7,10,0.1,0.60,50,22,6.6815,0.8722,0.1102,48701.0106,0.0919
8,15,0.1,0.40,50,22,6.8645,0.8854,0.0900,72132.4875,0.0950
9,15,0.1,0.45,50,22,6.7469,0.8856,0.0905,72253.2729,0.0970


#### Observation :
- This block ranks all valid UMAP + DBSCAN parameter combinations using the previously calculated `matching_error`.
- The results are sorted from lowest to highest matching error, so the combinations closest to the Ju and Dubin benchmark appear first.
- It displays the top 10 configurations with their UMAP parameters (`n_neighbors`, `min_dist`) and DBSCAN parameters (`epsilon`, `min_samples`).
- It also shows the resulting number of clusters, outlier percentage, Silhouette Score, DBI, CHI, and matching error.
- This allows the strongest UMAP configurations to be compared directly before selecting the final one.
- Overall, this block identifies the UMAP + DBSCAN parameter combinations that most closely match the published clustering characteristics.

### Show the Best UMAP + DBSCAN Hyperparameters

In [25]:
best_umap = umap_ranked.iloc[0]

print(
    "Best UMAP + DBSCAN Hyperparameters"
)

print("-" * 45)

print(
    "n_neighbors:",
    int(best_umap["n_neighbors"])
)

print(
    "min_dist:",
    best_umap["min_dist"]
)

print(
    "Epsilon:",
    best_umap["epsilon"]
)

print(
    "Min samples:",
    int(best_umap["min_samples"])
)

print()

print("Clustering result")

print(
    "Clusters:",
    int(best_umap["clusters"])
)

print(
    "Outlier percentage:",
    round(best_umap["outlier_percentage"], 2),
    "%"
)

print(
    "Silhouette:",
    round(best_umap["silhouette"], 4)
)

print(
    "Davies-Bouldin Index:",
    round(best_umap["dbi"], 4)
)

print(
    "Calinski-Harabasz Index:",
    round(best_umap["chi"], 2)
)

print(
    "Matching error:",
    round(best_umap["matching_error"], 4)
)

Best UMAP + DBSCAN Hyperparameters
---------------------------------------------
n_neighbors: 10
min_dist: 0.1
Epsilon: 0.35
Min samples: 50

Clustering result
Clusters: 22
Outlier percentage: 6.84 %
Silhouette: 0.87
Davies-Bouldin Index: 0.1091
Calinski-Harabasz Index: 48604.16
Matching error: 0.0886


#### Observation :
- This block selects the best UMAP + DBSCAN configuration from the ranked tuning results.
- It takes the parameter combination with the lowest matching error compared with the Ju and Dubin benchmark.
- It reports the selected UMAP settings: `n_neighbors` and `min_dist`, along with DBSCAN `epsilon` and `min_samples`.
- It also displays the resulting number of clusters, outlier percentage, Silhouette Score, Davies–Bouldin Index, and Calinski–Harabasz Index.
- The final `matching error` shows how closely this selected configuration reproduces the published clustering characteristics.
- Overall, this block identifies and summarizes the final best-performing UMAP + DBSCAN hyperparameters from the tuning process.

### Save UMAP Hyperparameter Tuning Results

In [26]:
# Save every tested UMAP + DBSCAN parameter combination
umap_tuning_df.to_csv(
    RESULTS_DIR / "umap_tuning_results.csv",
    index=False
)

# Save the 10 UMAP parameter combinations
# with the smallest matching error
umap_ranked.head(10).to_csv(
    RESULTS_DIR / "umap_top_parameters.csv",
    index=False
)

print("Saved UMAP tuning results.")

Saved UMAP tuning results.


#### Observation :
- This block saves the complete UMAP + DBSCAN hyperparameter tuning results into CSV files.
- `umap_tuning_results.csv` stores all tested combinations of UMAP and DBSCAN parameters with their clustering metrics.
- A second file, `umap_top_parameters.csv`, stores the top 10 configurations ranked by the lowest matching error.
- Saving both files keeps the full tuning history while also providing quick access to the best-performing parameter combinations.
- Overall, this block preserves the UMAP tuning results for later comparison, reporting, and reuse.

## PyTorch VAE + DBSCAN Hyperparameter Tuning

### Convert the Feature Matrix to a PyTorch Tensor

In [27]:
# Convert the NumPy clustering matrix into a PyTorch tensor
X_tensor = torch.tensor(
    X,
    dtype=torch.float32
)

print(
    "PyTorch feature matrix shape:",
    X_tensor.shape
)

PyTorch feature matrix shape: torch.Size([7648, 80])


#### Observation :
- This block converts the feature matrix `X` from a NumPy array into a PyTorch tensor.
- The data type is set to `float32`, which is the standard format used for neural network training in PyTorch.
- The resulting tensor, `X_tensor`, will be used as the input for the Variational Autoencoder (VAE).
- The tensor shape is printed to confirm that the number of patient records and features remains unchanged after conversion.
- Overall, this block prepares the patient feature data in the correct PyTorch format for VAE training.

### Define the PyTorch VAE

In [28]:
class VAE(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_1,
        hidden_2,
        latent_dim=2
    ):

        super().__init__()

        # -------------------------
        # Encoder
        # -------------------------

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.ReLU(),

            nn.Linear(hidden_1, hidden_2),
            nn.ReLU()
        )

        # Mean of the latent distribution
        self.z_mean = nn.Linear(
            hidden_2,
            latent_dim
        )

        # Log variance of the latent distribution
        self.z_log_var = nn.Linear(
            hidden_2,
            latent_dim
        )

        # -------------------------
        # Decoder
        # -------------------------

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_2),
            nn.ReLU(),

            nn.Linear(hidden_2, hidden_1),
            nn.ReLU(),

            # Original features are Min-Max scaled,
            # so sigmoid keeps reconstructed values near 0-1
            nn.Linear(hidden_1, input_dim),
            nn.Sigmoid()
        )


    def encode(self, x):

        # Learn the hidden representation
        hidden = self.encoder(x)

        # Estimate mean and log variance
        mean = self.z_mean(hidden)
        log_var = self.z_log_var(hidden)

        return mean, log_var


    def reparameterize(
        self,
        mean,
        log_var
    ):

        # Standard deviation of the latent distribution
        std = torch.exp(
            0.5 * log_var
        )

        # Random noise with the same shape
        epsilon = torch.randn_like(std)

        # Reparameterization trick
        z = mean + epsilon * std

        return z


    def decode(self, z):

        return self.decoder(z)


    def forward(self, x):

        # Encode patient features
        mean, log_var = self.encode(x)

        # Sample latent representation
        z = self.reparameterize(
            mean,
            log_var
        )

        # Reconstruct patient features
        reconstruction = self.decode(z)

        return (
            reconstruction,
            mean,
            log_var
        )

#### Observation :
- This block defines the Variational Autoencoder (VAE) architecture used to learn a lower-dimensional representation of the patient data.
- The encoder gradually compresses the original clinical features through two hidden layers using `ReLU` activation.
- Instead of producing a single fixed latent value, the encoder learns a mean and log-variance, which describe the distribution of each patient in the latent space.
- The reparameterization step samples from this distribution while still allowing the network to be trained through backpropagation.
- The latent space is fixed to $2$ dimensions, making it suitable for later DBSCAN clustering and visualization.
- The decoder reconstructs the original patient features from the latent representation.
- A `Sigmoid` activation is used at the output because the input features were Min-Max scaled, keeping reconstructed values close to the $0–1$ range.
- The `forward()` method connects the full process: encode → sample latent representation → reconstruct the input.
- Overall, this block builds the VAE that will learn a compact $2$-dimensional representation of patient characteristics for subsequent DBSCAN clustering.

### Define the VAE Loss

In [29]:
def vae_loss(
    reconstruction,
    original,
    mean,
    log_var,
    beta
):

    # Reconstruction loss measures how accurately
    # the VAE reproduces the original patient features
    reconstruction_loss = nn.functional.mse_loss(
        reconstruction,
        original,
        reduction="sum"
    )

    # KL divergence regularizes the latent representation
    kl_loss = -0.5 * torch.sum(
        1
        + log_var
        - mean.pow(2)
        - log_var.exp()
    )

    # beta determines how strongly the KL term
    # influences the VAE representation
    total_loss = (
        reconstruction_loss
        + beta * kl_loss
    )

    return total_loss

#### Observation :
- This block defines the loss function used to train the VAE.
- The reconstruction loss uses Mean Squared Error (MSE) to measure how closely the reconstructed patient features match the original input features.
- The KL-divergence loss encourages the latent representation to follow a structured probability distribution rather than becoming irregular or overly fitted to the training data.
- The `beta` parameter controls how much importance is given to this latent-space regularization.
- The final VAE loss combines both parts: reconstruction accuracy + beta-weighted KL divergence.
- Overall, this block balances accurate reconstruction of patient data with a well-structured latent space that can later support more meaningful DBSCAN clustering.

### Create the VAE Training Function

In [30]:
def train_vae(
    X_tensor,
    hidden_1,
    hidden_2,
    learning_rate,
    beta,
    batch_size,
    epochs
):

    # Reset seeds before each model
    # so architecture comparisons remain reproducible
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            RANDOM_STATE
        )

    # Create a fresh VAE
    model = VAE(
        input_dim=X_tensor.shape[1],
        hidden_1=hidden_1,
        hidden_2=hidden_2,
        latent_dim=2
    ).to(device)

    # Adam optimizer is commonly used
    # for autoencoder training
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    # Create the training DataLoader
    dataset = TensorDataset(
        X_tensor
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    # Put the network in training mode
    model.train()

    for epoch in range(epochs):

        total_epoch_loss = 0

        for batch in loader:

            # TensorDataset returns a tuple
            batch_x = batch[0].to(device)

            # Clear previous gradients
            optimizer.zero_grad()

            # Forward pass through VAE
            reconstruction, mean, log_var = (
                model(batch_x)
            )

            # Calculate VAE loss
            loss = vae_loss(
                reconstruction,
                batch_x,
                mean,
                log_var,
                beta
            )

            # Calculate gradients
            loss.backward()

            # Update model weights
            optimizer.step()

            total_epoch_loss += loss.item()

    return model

#### Observation :
- This block defines the function used to train each VAE configuration during hyperparameter tuning.
- Random seeds are reset before every training run so different VAE configurations can be compared under consistent conditions.
- A new VAE model is created using the selected hidden-layer sizes and moved to the available CPU or GPU.
- The Adam optimizer is used with the learning rate being tested.
- The patient data is divided into batches using a `DataLoader`, with shuffling enabled during training.
- For each epoch, the model performs a forward pass, calculates the VAE loss, computes gradients, and updates the network weights.
- After all epochs are completed, the trained model is returned for extracting its latent representation.
- Overall, this block provides a consistent training process for every VAE hyperparameter combination so their resulting patient representations can be fairly compared.

### Create a Function to Extract the VAE Latent Representation

In [31]:
def get_latent_representation(
    model,
    X_tensor
):

    # Evaluation mode disables training behavior
    model.eval()

    with torch.no_grad():

        # Move the complete matrix to the selected device
        X_device = X_tensor.to(device)

        # Use the mean of the latent distribution.
        # This avoids adding random sampling noise to clustering.
        mean, _ = model.encode(
            X_device
        )

        latent = (
            mean
            .cpu()
            .numpy()
        )

    return latent

#### Observation :
- This block defines a function to extract the VAE’s latent representation after training.
- The model is switched to evaluation mode, ensuring that no training behavior affects the output.
- `torch.no_grad()` is used because gradients are not needed during latent-feature extraction, which reduces memory and computation.
- The complete patient feature matrix is moved to the selected CPU or GPU device.
- The function uses the mean (`mean`) of the latent distribution instead of randomly sampling from it.
- Using the latent mean removes sampling noise and gives a stable, reproducible representation for DBSCAN clustering.
- The latent values are moved back to the CPU and converted into a NumPy array.
- Overall, this block extracts a stable low-dimensional patient representation from the trained VAE for use in DBSCAN clustering.

### Define VAE Parameters to Test

In [32]:
# Two simple architectures are enough for the initial search
vae_hidden_configs = [
    (32, 16),
    (64, 32),
    (128, 64),
    (256, 128)
]

# Test two learning rates
vae_learning_rates = [
    0.0001,
    0.0002,
    0.0003,
    0.0004,
    0.0005,
    0.0006,
    0.0007,
    0.0008,
    0.0009,
    0.0010
]

# beta controls VAE latent-space regularization
vae_beta_values = [
    0.001,
    0.01,
    0.1
]

# Keep these fixed to avoid unnecessary model complexity
VAE_BATCH_SIZE = 128
VAE_EPOCHS = 50
VAE_LATENT_DIM = 2

print(
    "VAE hidden configurations:",
    vae_hidden_configs
)

print(
    "Learning rates:",
    vae_learning_rates
)

print(
    "Beta values:",
    vae_beta_values
)

print(
    "Batch size:",
    VAE_BATCH_SIZE
)

print(
    "Epochs:",
    VAE_EPOCHS
)

VAE hidden configurations: [(32, 16), (64, 32), (128, 64), (256, 128)]
Learning rates: [0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008, 0.0009, 0.001]
Beta values: [0.001, 0.01, 0.1]
Batch size: 128
Epochs: 50


#### Observation :
- This block defines the VAE hyperparameter search space.
- Four encoder-decoder hidden-layer configurations are tested, ranging from smaller $(32, 16)$ to larger $(256, 128)$ networks.
- Learning rates from $0.0001$ to $0.0010$ are tested to find a suitable speed for model optimization.
- Three `beta` values $(0.001, 0.01, 0.1)$ are tested to control the strength of latent-space regularization.
- The batch size is fixed at $128$, the model is trained for $50$ epochs, and the latent space is kept at $2$ dimensions.
- Keeping these settings fixed reduces unnecessary complexity while allowing the main VAE parameters to be compared systematically.
- Overall, this block defines the VAE configurations that will be tested to find a latent representation most suitable for DBSCAN clustering.

### Define VAE DBSCAN Parameters to Test

In [33]:
# VAE latent coordinates exist on their own numerical scale.
# This epsilon range includes the value around 0.38
# used in our previous VAE replication.
vae_eps_values = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70
]

print(
    "VAE epsilon values:",
    vae_eps_values
)

print(
    "DBSCAN min samples:",
    dbscan_min_samples_values
)

VAE epsilon values: [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]
DBSCAN min samples: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


#### Observation :
- This block defines the DBSCAN `eps` values that will be tested on the VAE latent representation.
- The values range from $0.20$ to $0.70$, covering different neighborhood distances for identifying clusters.
- This range is chosen specifically for the numerical scale of the VAE latent space, which differs from t-SNE and UMAP.
- The range also includes values around $0.38$, which were useful in the previous VAE replication.
- The existing DBSCAN `min_samples` values are reused, allowing different combinations of neighborhood radius and minimum cluster density to be tested.
- Overall, this block defines an appropriate DBSCAN search range for clustering the VAE-generated patient representation.

### Run PyTorch VAE + DBSCAN Parameter Search

In [34]:
vae_tuning_results = []

for hidden_1, hidden_2 in vae_hidden_configs:

    for learning_rate in vae_learning_rates:

        for beta in vae_beta_values:

            print(
                "Training VAE:",
                f"hidden=({hidden_1}, {hidden_2}),",
                f"lr={learning_rate},",
                f"beta={beta}"
            )

            # Train one VAE configuration
            vae_model = train_vae(
                X_tensor=X_tensor,
                hidden_1=hidden_1,
                hidden_2=hidden_2,
                learning_rate=learning_rate,
                beta=beta,
                batch_size=VAE_BATCH_SIZE,
                epochs=VAE_EPOCHS
            )

            # Extract the stable 2-D latent means
            vae_embedding = (
                get_latent_representation(
                    vae_model,
                    X_tensor
                )
            )

            # Test DBSCAN parameters
            # on this VAE representation
            for eps in vae_eps_values:

                for min_samples in dbscan_min_samples_values:

                    dbscan = DBSCAN(
                        eps=eps,
                        min_samples=min_samples,
                        metric="euclidean"
                    )

                    labels = dbscan.fit_predict(
                        vae_embedding
                    )

                    # Exclude DBSCAN noise
                    valid_mask = labels != -1

                    valid_embedding = (
                        vae_embedding[
                            valid_mask
                        ]
                    )

                    valid_labels = (
                        labels[
                            valid_mask
                        ]
                    )

                    n_clusters = len(
                        np.unique(
                            valid_labels
                        )
                    )

                    n_outliers = np.sum(
                        labels == -1
                    )

                    outlier_percentage = (
                        n_outliers
                        / len(labels)
                    ) * 100

                    if n_clusters >= 2:

                        try:

                            sample_size = min(
                                SILHOUETTE_SAMPLE_SIZE,
                                len(valid_embedding)
                            )

                            silhouette = silhouette_score(
                                valid_embedding,
                                valid_labels,
                                sample_size=sample_size,
                                random_state=RANDOM_STATE
                            )

                            dbi = davies_bouldin_score(
                                valid_embedding,
                                valid_labels
                            )

                            chi = calinski_harabasz_score(
                                valid_embedding,
                                valid_labels
                            )

                        except ValueError:

                            silhouette = np.nan
                            dbi = np.nan
                            chi = np.nan

                    else:

                        silhouette = np.nan
                        dbi = np.nan
                        chi = np.nan

                    vae_tuning_results.append({
                        "hidden_1": hidden_1,
                        "hidden_2": hidden_2,
                        "learning_rate": learning_rate,
                        "beta": beta,
                        "batch_size": VAE_BATCH_SIZE,
                        "epochs": VAE_EPOCHS,
                        "epsilon": eps,
                        "min_samples": min_samples,
                        "clusters": n_clusters,
                        "outliers": n_outliers,
                        "outlier_percentage": outlier_percentage,
                        "silhouette": silhouette,
                        "dbi": dbi,
                        "chi": chi
                    })

            # Free the trained model before the next configuration
            del vae_model

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

Training VAE: hidden=(32, 16), lr=0.0001, beta=0.001
Training VAE: hidden=(32, 16), lr=0.0001, beta=0.01
Training VAE: hidden=(32, 16), lr=0.0001, beta=0.1
Training VAE: hidden=(32, 16), lr=0.0002, beta=0.001
Training VAE: hidden=(32, 16), lr=0.0002, beta=0.01
Training VAE: hidden=(32, 16), lr=0.0002, beta=0.1
Training VAE: hidden=(32, 16), lr=0.0003, beta=0.001
Training VAE: hidden=(32, 16), lr=0.0003, beta=0.01
Training VAE: hidden=(32, 16), lr=0.0003, beta=0.1
Training VAE: hidden=(32, 16), lr=0.0004, beta=0.001
Training VAE: hidden=(32, 16), lr=0.0004, beta=0.01
Training VAE: hidden=(32, 16), lr=0.0004, beta=0.1
Training VAE: hidden=(32, 16), lr=0.0005, beta=0.001
Training VAE: hidden=(32, 16), lr=0.0005, beta=0.01
Training VAE: hidden=(32, 16), lr=0.0005, beta=0.1
Training VAE: hidden=(32, 16), lr=0.0006, beta=0.001
Training VAE: hidden=(32, 16), lr=0.0006, beta=0.01
Training VAE: hidden=(32, 16), lr=0.0006, beta=0.1
Training VAE: hidden=(32, 16), lr=0.0007, beta=0.001
Training VA

#### Observation :
- This block performs the complete VAE + DBSCAN hyperparameter search.
- It trains a separate VAE for every combination of hidden-layer size, learning rate, and beta value.
- After each VAE is trained, the $2$-dimensional latent representation of the patient data is extracted.
- DBSCAN is then tested on that representation using every combination of `eps` and `min_samples`.
- DBSCAN noise points labeled $-1$ are excluded before evaluating cluster quality.
- For each valid clustering result, the block calculates the number of clusters, outlier percentage, Silhouette Score, Davies–Bouldin Index, and Calinski–Harabasz Index.
- Configurations producing fewer than two clusters are assigned `NaN` evaluation scores.
- All VAE and DBSCAN settings, together with their clustering results, are stored in `vae_tuning_results`.
- After each VAE configuration, the trained model is deleted and GPU memory is cleared to reduce memory usage during the large search.
- Overall, this block systematically tests the VAE architecture and DBSCAN settings to identify which combination produces the strongest patient clustering results.

### Create the VAE Results Table

In [35]:
vae_tuning_df = pd.DataFrame(
    vae_tuning_results
)

print(
    "Parameter combinations tested:",
    len(vae_tuning_df)
)

vae_tuning_df.head()

Parameter combinations tested: 13200


,hidden_1,hidden_2,learning_rate,beta,batch_size,epochs,epsilon,min_samples,clusters,outliers,outlier_percentage,silhouette,dbi,chi
0,32,16,0.0001,0.001,128,50,0.2,10,1,4,0.052301,NaN,NaN,NaN
1,32,16,0.0001,0.001,128,50,0.2,20,1,6,0.078452,NaN,NaN,NaN
2,32,16,0.0001,0.001,128,50,0.2,30,1,11,0.143828,NaN,NaN,NaN
3,32,16,0.0001,0.001,128,50,0.2,40,1,13,0.169979,NaN,NaN,NaN
4,32,16,0.0001,0.001,128,50,0.2,50,1,16,0.209205,NaN,NaN,NaN


#### Observation :
- This block converts all VAE + DBSCAN tuning results into a pandas DataFrame called `vae_tuning_df`.
- Each row represents one tested combination of VAE architecture, learning rate, beta, and DBSCAN parameters.
- It prints the total number of parameter combinations tested, confirming the size of the VAE tuning search.
- `head()` displays the first few rows to check that the tuning results and clustering metrics were stored correctly.
- Overall, this block organizes the complete VAE tuning results into a structured table for later comparison and ranking.

### Calculate VAE Similarity to Ju and Dubin


In [36]:
vae_tuning_df = vae_tuning_df.dropna(
    subset=[
        "silhouette",
        "dbi",
        "chi"
    ]
).copy()


vae_tuning_df["cluster_error"] = (
    abs(
        vae_tuning_df["clusters"]
        - VAE_TARGET_CLUSTERS
    )
    / VAE_TARGET_CLUSTERS
)


vae_tuning_df["silhouette_error"] = (
    abs(
        vae_tuning_df["silhouette"]
        - VAE_TARGET_SILHOUETTE
    )
    / VAE_TARGET_SILHOUETTE
)


vae_tuning_df["dbi_error"] = (
    abs(
        vae_tuning_df["dbi"]
        - VAE_TARGET_DBI
    )
    / VAE_TARGET_DBI
)


vae_tuning_df["chi_error"] = (
    abs(
        vae_tuning_df["chi"]
        - VAE_TARGET_CHI
    )
    / VAE_TARGET_CHI
)


vae_tuning_df["matching_error"] = (
    vae_tuning_df[
        [
            "cluster_error",
            "silhouette_error",
            "dbi_error",
            "chi_error"
        ]
    ]
    .mean(axis=1)
)

#### Observation :
- This block removes VAE + DBSCAN results where the Silhouette Score, DBI, or CHI could not be calculated.
- It compares every valid result with the target VAE clustering results from Ju and Dubin.
- Relative errors are calculated for the number of clusters, Silhouette Score, DBI, and CHI.
- These four errors are averaged into one `matching_error` value.
- A lower `matching_error` means that the tested VAE + DBSCAN configuration is closer to the published benchmark.
- Overall, this block creates a single comparison score to identify which VAE + DBSCAN configuration best reproduces the target clustering results.

### Rank VAE Parameter Combinations

In [37]:
vae_ranked = (
    vae_tuning_df
    .sort_values("matching_error")
    .reset_index(drop=True)
)

vae_ranked[
    [
        "hidden_1",
        "hidden_2",
        "learning_rate",
        "beta",
        "epsilon",
        "min_samples",
        "clusters",
        "outlier_percentage",
        "silhouette",
        "dbi",
        "chi",
        "matching_error"
    ]
].head(10).round(4)

,hidden_1,hidden_2,learning_rate,beta,epsilon,min_samples,clusters,outlier_percentage,silhouette,dbi,chi,matching_error
0,128,64,0.0007,0.1,0.20,90,12,4.4979,0.8794,0.2123,135198.7479,0.0386
1,128,64,0.0007,0.1,0.20,80,12,4.4979,0.8794,0.2123,135198.7479,0.0386
2,128,64,0.0007,0.1,0.20,100,12,4.4979,0.8794,0.2123,135198.7479,0.0386
3,128,64,0.0007,0.1,0.20,70,12,4.1972,0.8788,0.2133,129074.2792,0.0510
4,128,64,0.0007,0.1,0.20,60,12,4.1710,0.8772,0.2143,128550.3168,0.0537
5,256,128,0.0001,0.1,0.20,70,12,0.0000,0.8827,0.2036,161589.0871,0.0613
6,256,128,0.0001,0.1,0.20,60,12,0.0000,0.8827,0.2036,161589.0871,0.0613
7,256,128,0.0001,0.1,0.20,50,12,0.0000,0.8827,0.2036,161589.0871,0.0613
8,256,128,0.0001,0.1,0.30,70,12,0.0000,0.8827,0.2036,161589.0871,0.0613
9,256,128,0.0001,0.1,0.25,20,12,0.0000,0.8827,0.2036,161589.0871,0.0613


#### Observation :
- This block ranks all valid VAE + DBSCAN parameter combinations using the `matching_error`.
- The results are sorted from lowest to highest matching error, so the configurations closest to the Ju and Dubin benchmark appear first.
- It displays the top 10 configurations with their VAE settings: hidden-layer sizes, learning rate, and beta.
- It also includes the DBSCAN settings: `epsilon` and `min_samples`.
- For each configuration, it shows the resulting number of clusters, outlier percentage, Silhouette Score, DBI, CHI, and matching error.
- Overall, this block identifies the VAE + DBSCAN configurations that most closely match the published clustering results.

### Show the Best PyTorch VAE + DBSCAN Hyperparameters

In [38]:
best_vae = vae_ranked.iloc[0]

print(
    "Best PyTorch VAE + DBSCAN Hyperparameters"
)

print("-" * 50)

print(
    "Hidden layer 1:",
    int(best_vae["hidden_1"])
)

print(
    "Hidden layer 2:",
    int(best_vae["hidden_2"])
)

print(
    "Latent dimensions:",
    VAE_LATENT_DIM
)

print(
    "Learning rate:",
    best_vae["learning_rate"]
)

print(
    "Beta:",
    best_vae["beta"]
)

print(
    "Batch size:",
    int(best_vae["batch_size"])
)

print(
    "Epochs:",
    int(best_vae["epochs"])
)

print(
    "Epsilon:",
    best_vae["epsilon"]
)

print(
    "Min samples:",
    int(best_vae["min_samples"])
)

print()

print("Clustering result")

print(
    "Clusters:",
    int(best_vae["clusters"])
)

print(
    "Outlier percentage:",
    round(
        best_vae["outlier_percentage"],
        2
    ),
    "%"
)

print(
    "Silhouette:",
    round(best_vae["silhouette"], 4)
)

print(
    "Davies-Bouldin Index:",
    round(best_vae["dbi"], 4)
)

print(
    "Calinski-Harabasz Index:",
    round(best_vae["chi"], 2)
)

print(
    "Matching error:",
    round(
        best_vae["matching_error"],
        4
    )
)

Best PyTorch VAE + DBSCAN Hyperparameters
--------------------------------------------------
Hidden layer 1: 128
Hidden layer 2: 64
Latent dimensions: 2
Learning rate: 0.0007
Beta: 0.1
Batch size: 128
Epochs: 50
Epsilon: 0.2
Min samples: 90

Clustering result
Clusters: 12
Outlier percentage: 4.5 %
Silhouette: 0.8794
Davies-Bouldin Index: 0.2123
Calinski-Harabasz Index: 135198.75
Matching error: 0.0386


#### Observation :
- This block selects the best VAE + DBSCAN configuration from the ranked tuning results.
- It takes the configuration with the lowest `matching_error`, meaning it is the closest overall match to the Ju and Dubin benchmark.
- It reports the selected hidden-layer architecture, learning rate, beta, DBSCAN `epsilon`, and `min_samples`.
- It also displays the resulting number of clusters, outlier percentage, Silhouette Score, Davies–Bouldin Index, and Calinski–Harabasz Index.
- This provides the final tuned VAE settings that will be used in the later clustering analysis.
- Overall, this block identifies and summarizes the final best-performing VAE + DBSCAN hyperparameter combination.

## Save VAE Hyperparameter Tuning Results

In [39]:
# Save every tested PyTorch VAE + DBSCAN parameter combination
vae_tuning_df.to_csv(
    RESULTS_DIR / "vae_tuning_results.csv",
    index=False
)

# Save the 10 parameter combinations
# that are closest to the Ju and Dubin VAE results
vae_ranked.head(10).to_csv(
    RESULTS_DIR / "vae_top_parameters.csv",
    index=False
)

print("Saved VAE tuning results.")

Saved VAE tuning results.


#### Observation :
- This block saves the complete VAE + DBSCAN hyperparameter tuning results to `vae_tuning_results.csv`.
- It also saves the top 10 VAE configurations with the lowest matching error to `vae_top_parameters.csv`.
- The first file preserves every tested VAE and DBSCAN combination, while the second provides quick access to the configurations that most closely match the Ju and Dubin benchmark.
- The confirmation message verifies that the VAE tuning results were successfully stored.
- Overall, this block preserves both the full VAE tuning experiment and the best-performing parameter combinations for later analysis and reuse.

## Best Hyperparameters Summary

### Create the Best Hyperparameter Table

In [40]:
best_parameters = pd.DataFrame({

    "method": [
        "t-SNE + DBSCAN",
        "UMAP + DBSCAN",
        "PyTorch VAE + DBSCAN"
    ],

    "embedding_parameters": [

        (
            f"perplexity="
            f"{int(best_tsne['perplexity'])}"
        ),

        (
            f"n_neighbors="
            f"{int(best_umap['n_neighbors'])}, "
            f"min_dist="
            f"{best_umap['min_dist']}"
        ),

        (
            f"hidden="
            f"({int(best_vae['hidden_1'])}, "
            f"{int(best_vae['hidden_2'])}), "
            f"lr={best_vae['learning_rate']}, "
            f"beta={best_vae['beta']}"
        )
    ],

    "epsilon": [
        best_tsne["epsilon"],
        best_umap["epsilon"],
        best_vae["epsilon"]
    ],

    "min_samples": [
        int(best_tsne["min_samples"]),
        int(best_umap["min_samples"]),
        int(best_vae["min_samples"])
    ],

    "clusters": [
        int(best_tsne["clusters"]),
        int(best_umap["clusters"]),
        int(best_vae["clusters"])
    ],

    "silhouette": [
        best_tsne["silhouette"],
        best_umap["silhouette"],
        best_vae["silhouette"]
    ],

    "dbi": [
        best_tsne["dbi"],
        best_umap["dbi"],
        best_vae["dbi"]
    ],

    "chi": [
        best_tsne["chi"],
        best_umap["chi"],
        best_vae["chi"]
    ],

    "matching_error": [
        best_tsne["matching_error"],
        best_umap["matching_error"],
        best_vae["matching_error"]
    ]
})

best_parameters.round(4)

,method,embedding_parameters,epsilon,min_samples,clusters,silhouette,dbi,chi,matching_error
0,t-SNE + DBSCAN,perplexity=80,3.00,40,22,0.6695,0.2766,8775.6767,0.0326
1,UMAP + DBSCAN,"n_neighbors=10, min_dist=0.1",0.35,50,22,0.8700,0.1091,48604.1647,0.0886
2,PyTorch VAE + DBSCAN,"hidden=(128, 64), lr=0.0007, beta=0.1",0.20,90,12,0.8794,0.2123,135198.7479,0.0386


#### Observation :
- This block creates a single summary table of the best hyperparameters selected for all three methods: t-SNE + DBSCAN, UMAP + DBSCAN, and VAE + DBSCAN.
- For each method, it records the selected dimensionality-reduction parameters together with DBSCAN `epsilon` and `min_samples`.
- It also includes the resulting number of clusters, Silhouette Score, DBI, CHI, and matching error.
- This allows the final tuned configurations of all three approaches to be compared side by side in one place.
- The table is rounded to four decimal places for clearer presentation.
- Overall, this block brings together the final selected settings and clustering performance of all three methods into one comparison table.

### Save the Best Hyperparameters

In [41]:
# Save the final selected hyperparameters for all three methods.
# These values can later be copied directly into the
# t-SNE, UMAP, and VAE clustering notebooks.
best_parameters.to_csv(
    RESULTS_DIR / "best_hyperparameters.csv",
    index=False
)

print("Saved best hyperparameters.")

Saved best hyperparameters.


#### Observation :
- This block saves the final selected hyperparameters for all three clustering approaches into `best_hyperparameters.csv`.
- The file contains the chosen settings for t-SNE + DBSCAN, UMAP + DBSCAN, and VAE + DBSCAN.
- Saving these values in one file makes it easy to reuse the exact tuned parameters in the later clustering notebooks.
- The confirmation message verifies that the final hyperparameter file was saved successfully.
- Overall, this block creates a single reusable file containing the final hyperparameters selected from the complete tuning process.

### Compare Best Results with Ju and Dubin

In [42]:
best_vs_paper = pd.DataFrame({

    "method": [
        "t-SNE + DBSCAN",
        "UMAP + DBSCAN",
        "VAE + DBSCAN"
    ],

    "our_clusters": [
        int(best_tsne["clusters"]),
        int(best_umap["clusters"]),
        int(best_vae["clusters"])
    ],

    "paper_clusters": [
        22,
        22,
        12
    ],

    "our_silhouette": [
        best_tsne["silhouette"],
        best_umap["silhouette"],
        best_vae["silhouette"]
    ],

    "paper_silhouette": [
        0.67,
        0.84,
        0.89
    ],

    "our_dbi": [
        best_tsne["dbi"],
        best_umap["dbi"],
        best_vae["dbi"]
    ],

    "paper_dbi": [
        0.25,
        0.09,
        0.19
    ],

    "our_chi": [
        best_tsne["chi"],
        best_umap["chi"],
        best_vae["chi"]
    ],

    "paper_chi": [
        8576.69,
        54413.23,
        138637.94
    ]
})

best_vs_paper.round(4)

,method,our_clusters,paper_clusters,our_silhouette,paper_silhouette,our_dbi,paper_dbi,our_chi,paper_chi
0,t-SNE + DBSCAN,22,22,0.6695,0.67,0.2766,0.25,8775.6767,8576.69
1,UMAP + DBSCAN,22,22,0.8700,0.84,0.1091,0.09,48604.1647,54413.23
2,VAE + DBSCAN,12,12,0.8794,0.89,0.2123,0.19,135198.7479,138637.94


#### Observation :
- This block creates a side-by-side comparison between the best results obtained in this notebook and the results reported by Ju and Dubin.
- The comparison is made for all three methods: t-SNE + DBSCAN, UMAP + DBSCAN, and VAE + DBSCAN.
- For each method, it compares the number of clusters, Silhouette Score, Davies–Bouldin Index, and Calinski–Harabasz Index.
- The notebook’s results are stored under `our_... columns`, while the published values are stored under `paper_... columns`.
- This makes it easy to see how closely the tuned models reproduce the clustering performance reported in the reference study.
- The results are rounded to four decimal places for easier interpretation.
- Overall, this block directly compares the final tuned clustering results with the published benchmark to evaluate how successfully the study has been reproduced.

### Save Best Results vs Ju and Dubin

In [43]:
# Save the final comparison between our tuned
# clustering results and the published paper values
best_vs_paper.to_csv(
    RESULTS_DIR / "best_results_vs_ju_dubin.csv",
    index=False
)

print("Saved comparison with Ju and Dubin.")

Saved comparison with Ju and Dubin.


#### Observation :
- This block saves the final comparison between the tuned clustering results and the Ju and Dubin benchmark.
- The comparison table is written to `best_results_vs_ju_dubin.csv`.
- The file contains results for t-SNE + DBSCAN, UMAP + DBSCAN, and VAE + DBSCAN.
- It preserves both the notebook’s results and the corresponding published values for cluster count, Silhouette Score, DBI, and CHI.
- The confirmation message verifies that the comparison file was saved successfully.
- Overall, this block stores the final benchmark comparison so the agreement between the tuned models and the published study can be reviewed and reported later.

### Show the Best Hyperparameters

In [44]:
print("FINAL HYPERPARAMETER SELECTION")
print("=" * 55)


print()
print("t-SNE + DBSCAN")
print("-" * 25)

print(
    "Perplexity:",
    int(best_tsne["perplexity"])
)

print(
    "Epsilon:",
    best_tsne["epsilon"]
)

print(
    "Min samples:",
    int(best_tsne["min_samples"])
)


print()
print("UMAP + DBSCAN")
print("-" * 25)

print(
    "n_neighbors:",
    int(best_umap["n_neighbors"])
)

print(
    "min_dist:",
    best_umap["min_dist"]
)

print(
    "Epsilon:",
    best_umap["epsilon"]
)

print(
    "Min samples:",
    int(best_umap["min_samples"])
)


print()
print("PyTorch VAE + DBSCAN")
print("-" * 25)

print(
    "Hidden layers:",
    (
        int(best_vae["hidden_1"]),
        int(best_vae["hidden_2"])
    )
)

print(
    "Latent dimensions:",
    VAE_LATENT_DIM
)

print(
    "Learning rate:",
    best_vae["learning_rate"]
)

print(
    "Beta:",
    best_vae["beta"]
)

print(
    "Batch size:",
    int(best_vae["batch_size"])
)

print(
    "Epochs:",
    int(best_vae["epochs"])
)

print(
    "Epsilon:",
    best_vae["epsilon"]
)

print(
    "Min samples:",
    int(best_vae["min_samples"])
)

FINAL HYPERPARAMETER SELECTION

t-SNE + DBSCAN
-------------------------
Perplexity: 80
Epsilon: 3.0
Min samples: 40

UMAP + DBSCAN
-------------------------
n_neighbors: 10
min_dist: 0.1
Epsilon: 0.35
Min samples: 50

PyTorch VAE + DBSCAN
-------------------------
Hidden layers: (128, 64)
Latent dimensions: 2
Learning rate: 0.0007
Beta: 0.1
Batch size: 128
Epochs: 50
Epsilon: 0.2
Min samples: 90


#### Observation :
- This block prints the final selected hyperparameters for all three clustering pipelines in a clear, readable format.
- For t-SNE + DBSCAN, it reports the selected `perplexity`, `epsilon`, and `min_samples`.
- For UMAP + DBSCAN, it reports the selected `n_neighbors`, `min_dist`, `epsilon`, and `min_samples`.
- For VAE + DBSCAN, it reports the selected `hidden-layer sizes`, `latent dimension`, `learning rate`, `beta`, `batch size`, `epochs`, `epsilon`, and `min_samples`.

### Verify Saved Result Files

In [45]:
# Display every file produced by this tuning notebook
saved_files = sorted(
    RESULTS_DIR.glob("*.csv")
)

print("Saved hyperparameter tuning files:")
print()

for file in saved_files:
    print(file.name)

Saved hyperparameter tuning files:

best_hyperparameters.csv
best_results_vs_ju_dubin.csv
tsne_top_parameters.csv
tsne_tuning_results.csv
umap_top_parameters.csv
umap_tuning_results.csv
vae_top_parameters.csv
vae_tuning_results.csv


#### Observation :
- This block lists all CSV files generated by the hyperparameter-tuning notebook.
- It searches the `RESULTS_DIR` folder for every file with the `.csv` extension.
- The files are sorted so the output is organized and easy to review.
- Each saved filename is printed, providing a final check that the t-SNE, UMAP, VAE, best-parameter, and benchmark-comparison results were successfully created.
- Overall, this block acts as a final verification step, confirming that all expected hyperparameter-tuning output files have been saved correctly.